# OCR benchmark: Tesseract vs EasyOCR vs RapidOCR vs FastPlate

Original experiment notebook. Compares OCR engines on 500 synthetic and 500 real Russian plate crops, records plate/character accuracy and latency. Outputs are kept as recorded on the training machine.


# Бенчмарк

## Установка зависимостей

In [ ]:
#!pip install pytesseract easyocr opencv-python-headless > /dev/null
!pip install rapidocr pyyaml
!pip install fast-plate-ocr
!pip install onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.6/252.6 MB 3.7 MB/s eta 0:00:00


## Импорт библиотек

In [ ]:
import re
import cv2
import random
import logging
import warnings
import numpy as np
import pandas as pd
from time import time
from pathlib import Path
from tqdm.notebook import tqdm
import yaml
import onnxruntime as ort

from rapidocr import RapidOCR
from fast_plate_ocr import LicensePlateRecognizer

warnings.filterwarnings(
    "ignore",
    message="'pin_memory' argument is set as true but no accelerator is found"
)
logging.getLogger("RapidOCR").setLevel(logging.ERROR)

## Конфигурация и загрузка моделей

In [ ]:
import cv2
import yaml
import random
import numpy as np
import pandas as pd
from time import time
from pathlib import Path
from tqdm.notebook import tqdm
import onnxruntime as ort

from rapidocr import RapidOCR
from fast_plate_ocr import LicensePlateRecognizer

# --- ПУТИ ---
IMG_DIR = Path("/content/drive/MyDrive/Dataset_car_plate/")
MY_ONNX_MODEL = "/content/drive/MyDrive/Dataset_car_plate/ru_plate_cct_xs.onnx"
PLATE_CONFIG_PATH = "/content/drive/MyDrive/Dataset_car_plate/ru_plate_config.yaml"

# --- ЗАГРУЗКА БАЗОВЫХ МОДЕЛЕЙ ---
print("Загрузка базовых моделей...")
rapid_reader = RapidOCR()
plate_reader_gl_orig = LicensePlateRecognizer("global-plates-mobile-vit-v2-model")

# --- ЗАГРУЗКА ТВОЕЙ МОДЕЛИ ---
if Path(MY_ONNX_MODEL).exists() and Path(PLATE_CONFIG_PATH).exists():
    try:
        # Читаем конфиг для получения алфавита
        with open(PLATE_CONFIG_PATH, 'r') as f:
            my_config = yaml.safe_load(f)

        my_alphabet = my_config['alphabet']
        my_pad_char = my_config.get('pad_char', '_')

        # Запускаем сессию ONNX на GPU
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        my_ort_session = ort.InferenceSession(MY_ONNX_MODEL, providers=providers)

        print(f"✅ Дообученная модель загружена: {Path(MY_ONNX_MODEL).name}")
        print(f"Провайдеры: {my_ort_session.get_providers()}")
    except Exception as e:
        print(f"❌ Ошибка загрузки локальной модели: {e}")
        my_ort_session = None
else:
    print(f"❌ ОШИБКА: Файл модели или конфига не найден!")
    my_ort_session = None

[INFO] 2026-02-22 16:47:12,786 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-22 16:47:12,817 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-22 16:47:12,819 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx


Загрузка базовых моделей...


[INFO] 2026-02-22 16:47:13,052 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-22 16:47:13,059 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-22 16:47:13,060 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-22 16:47:13,170 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-22 16:47:13,235 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-02-22 16:47:13,236 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx


*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:539 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(PySessionOptions&, const onnxruntime::ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Falling back to ['CUDAExecutionProvider', 'CPUExecutionProvider'] and retrying.
****************************************
✅ Дообученная модель загружена: ru_plate_cct_xs.onnx
Провайдеры: ['CPUExecutionProvider']


In [ ]:
!cp -r /content/drive/MyDrive/Dataset_car_plate /content/Dataset_car_plate_local

^C


## Функции обработки и нормализации

In [ ]:
def preprocess_plate(img_bgr: np.ndarray) -> np.ndarray:
    h, w = img_bgr.shape[:2]
    # Обрезаем края (от рамки номера)
    ch, cw = max(1, int(h * 0.08)), max(1, int(w * 0.04))
    img = img_bgr[ch:h - ch, cw:w - cw]
    h, w = img.shape[:2]
    # Ресайз по высоте
    scale = 150.0 / h
    img = cv2.resize(img, (int(w * scale), 150), interpolation=cv2.INTER_CUBIC)
    # Фильтры
    img = cv2.bilateralFilter(img, d=11, sigmaColor=17, sigmaSpace=17)
    kernel = np.array([[-1, -1, -1], [-1,  9, -1], [-1, -1, -1]])
    img = cv2.filter2D(img, -1, kernel)
    # Паддинг
    img = cv2.copyMakeBorder(img, 30, 30, 30, 30, cv2.BORDER_CONSTANT, value=[128, 128, 128])
    return img

In [ ]:
def normalize_plate_text(text: str) -> str:
    text = text.upper()
    text = text.replace('₽', 'P')
    text = re.sub(r'[^A-Z0-9АБВЕКМНОРСТУХ/\\]', '', text)
    replacements = {
        'А': 'A', 'В': 'B', 'Е': 'E', 'К': 'K', 'М': 'M', 'Н': 'H',
        'О': 'O', 'Р': 'P', 'С': 'C', 'Т': 'T', 'У': 'Y', 'Х': 'X'
    }
    for cyr, lat in replacements.items():
        text = text.replace(cyr, lat)
    return text

PLATE_LETTERS = set("ABEKMHOPCTYX")
DIGITS = set("0123456789")
PLATE_REGEX = re.compile(r"^[A-Z]\d{3}[A-Z]{2}\d{2,3}$")
LETTER_POS = {0, 4, 5}

LETTER_TO_DIGIT = {
    "O": "0", "Q": "0", "D": "0", "I": "1", "L": "1", "J": "1",
    "Z": "7", "T": "7", "V": "7", "Y": "7", "S": "5", "B": "8",
    "A": "4", "G": "6", "/": "7", "\\": "7",
}
TO_ALLOWED_LETTER = {
    "0": "O", "D": "O", "Q": "O", "4": "A", "7": "T", "I": "T",
    "1": "T", "8": "B", "5": "S", "S": "C", "U": "Y", "V": "Y",
    "N": "H", "R": "P", "3": "E", "F": "E", "J": "T", "W": "M",
    "/": "Y", "\\": "Y",
}
_FORCE_DIGIT_MAP = {
    "V": "7", "Y": "7", "Z": "7", "T": "7", "S": "5", "G": "6",
    "B": "8", "I": "1", "L": "1", "J": "1", "O": "0", "Q": "0",
    "D": "0", "A": "4",
}

def _to_digit(ch): return ch if ch in DIGITS else LETTER_TO_DIGIT.get(ch, ch)
def _to_letter(ch): return ch if ch in PLATE_LETTERS else TO_ALLOWED_LETTER.get(ch, ch)

def _fix_prefix6(raw6):
    chars = list(raw6)
    changes = 0
    for i, ch in enumerate(chars):
        new_ch = _to_letter(ch) if i in LETTER_POS else _to_digit(ch)
        if new_ch != ch: changes += 1
        chars[i] = new_ch
    ok = (chars[0] in PLATE_LETTERS and chars[1] in DIGITS and
          chars[2] in DIGITS and chars[3] in DIGITS and
          chars[4] in PLATE_LETTERS and chars[5] in PLATE_LETTERS)
    return "".join(chars), ok, changes

def _force_digits(s):
    result = list(s)
    for i in {1, 2, 3, 6, 7, 8}:
        if i < len(result) and result[i] in _FORCE_DIGIT_MAP:
            result[i] = _FORCE_DIGIT_MAP[result[i]]
    return "".join(result)

def postprocess_plate(text: str) -> str:
    if not text: return ""
    t = re.sub(r"[^0-9A-Z/\\]", "", text.upper())
    for noise in ["RUSSIA", "RUSE", "RUS", "PUS", "PVS", "RVS", "RU5", "PU5", "R0S", "KUS", "PLS"]:
        t = t.replace(noise, "")
    if len(t) > 6 and t[6] in ('/', '\\'): t = t[:6] + t[7:]
    if len(t) < 8: return t
    best = None
    for s in range(0, len(t) - 5):
        raw6 = t[s:s + 6]
        fixed6, ok, chg6 = _fix_prefix6(raw6)
        if not ok: continue
        tail = t[s + 6:]
        tail_digits, tail_changes = [], 0
        for ch in tail:
            d = _to_digit(ch)
            if d in DIGITS:
                tail_digits.append(d)
                if d != ch: tail_changes += 1
        for region_len in (3, 2):
            if len(tail_digits) >= region_len:
                cand = fixed6 + "".join(tail_digits[:region_len])
                if PLATE_REGEX.match(cand):
                    changes = chg6 + tail_changes
                    if best is None or changes < best[0]:
                        best = (changes, cand)
    return _force_digits(best[1]) if best else t

## Основной блок GPU

In [ ]:
import cv2
import random
import numpy as np
import pandas as pd
from time import time
from pathlib import Path
from tqdm.notebook import tqdm

# Используем скопированную локальную папку для МАКСИМАЛЬНОЙ скорости!
LOCAL_IMG_DIR = Path("/content/drive/MyDrive/Dataset_car_plate/")

# Получаем Ground Truth из имени файла
def get_gt_from_filename(img_path):
    parts = img_path.name.split(".")[0].split("_", 1)
    if len(parts) < 2: return None
    gt = normalize_plate_text(parts[1])
    return gt if (gt and PLATE_REGEX.match(gt)) else None

# Загружаем датасет
all_images = sorted(LOCAL_IMG_DIR.glob("*.png"))
annotated = []
for img_path in all_images:
    gt = get_gt_from_filename(img_path)
    if gt: annotated.append((img_path, gt))

# Берем 3000 случайных картинок
random.seed(42)
test_data = random.sample(annotated, min(500, len(annotated)))
print(f"Запуск бенчмарка на {len(test_data)} изображениях...")

# Метрики
def char_accuracy(pred, gt):
    if not gt: return 0.0
    return sum(p == g for p, g in zip(pred, gt)) / len(gt)

bench = {
    "RapidOCR": {"plate_correct": 0, "char_correct": 0.0, "total_chars": 0, "total_plates": 0, "time_sum": 0.0},
    "FastPlate_Global (Base)": {"plate_correct": 0, "char_correct": 0.0, "total_chars": 0, "total_plates": 0, "time_sum": 0.0},
    "FastPlate_Finetuned (New)": {"plate_correct": 0, "char_correct": 0.0, "total_chars": 0, "total_plates": 0, "time_sum": 0.0},
}

def update_stats(key, pred, gt, dt):
    bench[key]["total_plates"] += 1
    bench[key]["total_chars"] += len(gt)
    bench[key]["time_sum"] += dt
    bench[key]["plate_correct"] += int(pred == gt)
    bench[key]["char_correct"] += char_accuracy(pred, gt) * len(gt)

# --- ПРАВИЛЬНЫЙ ДЕКОДЕР CCE (Без удаления дубликатов!) ---
def cce_decode(preds, alphabet, pad_char):
    pred_indices = np.argmax(preds[0], axis=-1)
    pad_idx = alphabet.index(pad_char) if pad_char in alphabet else len(alphabet) - 1

    result = []
    for idx in pred_indices:
        if idx != pad_idx:
            result.append(alphabet[idx])

    return "".join(result)

# Проверяем, какой тип данных ожидает твой ONNX
expected_type = my_ort_session.get_inputs()[0].type if my_ort_session else ""

# Основной цикл
for img_path, gt in tqdm(test_data, desc="Тестирование"):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None: continue

    pre = preprocess_plate(img_bgr)
    img_rgb_rapid = cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)
    img_gray_raw = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 1. RapidOCR
    t0 = time()
    try:
        result = rapid_reader(img_rgb_rapid, use_det=False, use_cls=False)
        raw = "".join(result.txts).replace('/', '').replace('\\', '') if result and result.txts else ""
    except: raw = ""
    update_stats("RapidOCR", postprocess_plate(normalize_plate_text(raw)), gt, (time() - t0) * 1000)

    # 2. FastPlate_Global
    t0 = time()
    try:
        res = plate_reader_gl_orig.run(img_gray_raw)
        raw = res[0] if res else ""
    except: raw = ""
    update_stats("FastPlate_Global (Base)", postprocess_plate(normalize_plate_text(raw)), gt, (time() - t0) * 1000)

    # 3. FastPlate_Finetuned
    if my_ort_session:
        t0 = time()
        try:
            # Делаем ресайз методом cv2.INTER_LINEAR (как было при обучении)
            img_input = cv2.resize(img_gray_raw, (140, 70), interpolation=cv2.INTER_LINEAR)

            # Динамически подстраиваемся под то, как экспортировалась модель (uint8 или float32)
            if 'float' in expected_type:
                img_input = img_input.astype(np.float32) / 255.0
            else:
                img_input = img_input.astype(np.uint8)

            img_input = np.expand_dims(img_input, axis=(0, -1))

            # Прогон
            input_name = my_ort_session.get_inputs()[0].name
            preds = my_ort_session.run(None, {input_name: img_input})[0]

            # Правильное CCE декодирование
            raw = cce_decode(preds, my_alphabet, my_pad_char)
        except Exception as e:
            raw = ""

        update_stats("FastPlate_Finetuned (New)", postprocess_plate(normalize_plate_text(raw)), gt, (time() - t0) * 1000)

# Вывод результатов
print("\n" + "="*50)
print("🏆 ИТОГОВЫЕ РЕЗУЛЬТАТЫ БЕНЧМАРКА 🏆")
print("="*50)

results_df = []
for key, v in bench.items():
    if v["total_plates"] == 0: continue
    pt = v["total_plates"]

    plate_acc = (v["plate_correct"] / pt) * 100
    char_acc = (v["char_correct"] / v["total_chars"]) * 100
    avg_time = (v["time_sum"] / pt)

    results_df.append({
        "Модель": key,
        "Plate Acc (Номера целиком)": f"{plate_acc:.2f}%",
        "Char Acc (Посимвольно)": f"{char_acc:.2f}%",
        "Время на кадр (ms)": f"{avg_time:.2f}"
    })

df = pd.DataFrame(results_df)
display(df)

Запуск бенчмарка на 500 изображениях...


Тестирование:   0%|          | 0/500 [00:00<?, ?it/s]


🏆 ИТОГОВЫЕ РЕЗУЛЬТАТЫ БЕНЧМАРКА 🏆


,Модель,Plate Acc (Номера целиком),Char Acc (Посимвольно),Время на кадр (ms)
0,RapidOCR,93.60%,97.81%,38.34
1,FastPlate_Global (Base),52.00%,84.32%,7.62
2,FastPlate_Finetuned (New),98.20%,99.57%,2.35


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/17kcMRxAQrAjoyzU_G-eLvpdWWgzKH_iIGd3hEJH4T6c/edit#gid=0


## Основной блок CPU

In [ ]:
import cv2
import random
import numpy as np
import pandas as pd
from time import time
from pathlib import Path
from tqdm.notebook import tqdm

# Используем скопированную локальную папку для МАКСИМАЛЬНОЙ скорости!
LOCAL_IMG_DIR = Path("/content/drive/MyDrive/Dataset_car_plate/")

# Получаем Ground Truth из имени файла
def get_gt_from_filename(img_path):
    parts = img_path.name.split(".")[0].split("_", 1)
    if len(parts) < 2: return None
    gt = normalize_plate_text(parts[1])
    return gt if (gt and PLATE_REGEX.match(gt)) else None

# Загружаем датасет
all_images = sorted(LOCAL_IMG_DIR.glob("*.png"))
annotated = []
for img_path in all_images:
    gt = get_gt_from_filename(img_path)
    if gt: annotated.append((img_path, gt))

# Берем 3000 случайных картинок
random.seed(42)
test_data = random.sample(annotated, min(500, len(annotated)))
print(f"Запуск бенчмарка на {len(test_data)} изображениях...")

# Метрики
def char_accuracy(pred, gt):
    if not gt: return 0.0
    return sum(p == g for p, g in zip(pred, gt)) / len(gt)

bench = {
    "RapidOCR": {"plate_correct": 0, "char_correct": 0.0, "total_chars": 0, "total_plates": 0, "time_sum": 0.0},
    "FastPlate_Global (Base)": {"plate_correct": 0, "char_correct": 0.0, "total_chars": 0, "total_plates": 0, "time_sum": 0.0},
    "FastPlate_Finetuned (New)": {"plate_correct": 0, "char_correct": 0.0, "total_chars": 0, "total_plates": 0, "time_sum": 0.0},
}

def update_stats(key, pred, gt, dt):
    bench[key]["total_plates"] += 1
    bench[key]["total_chars"] += len(gt)
    bench[key]["time_sum"] += dt
    bench[key]["plate_correct"] += int(pred == gt)
    bench[key]["char_correct"] += char_accuracy(pred, gt) * len(gt)

# --- ПРАВИЛЬНЫЙ ДЕКОДЕР CCE (Без удаления дубликатов!) ---
def cce_decode(preds, alphabet, pad_char):
    pred_indices = np.argmax(preds[0], axis=-1)
    pad_idx = alphabet.index(pad_char) if pad_char in alphabet else len(alphabet) - 1

    result = []
    for idx in pred_indices:
        if idx != pad_idx:
            result.append(alphabet[idx])

    return "".join(result)

# Проверяем, какой тип данных ожидает твой ONNX
expected_type = my_ort_session.get_inputs()[0].type if my_ort_session else ""

# Основной цикл
for img_path, gt in tqdm(test_data, desc="Тестирование"):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None: continue

    pre = preprocess_plate(img_bgr)
    img_rgb_rapid = cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)
    img_gray_raw = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 1. RapidOCR
    t0 = time()
    try:
        result = rapid_reader(img_rgb_rapid, use_det=False, use_cls=False)
        raw = "".join(result.txts).replace('/', '').replace('\\', '') if result and result.txts else ""
    except: raw = ""
    update_stats("RapidOCR", postprocess_plate(normalize_plate_text(raw)), gt, (time() - t0) * 1000)

    # 2. FastPlate_Global
    t0 = time()
    try:
        res = plate_reader_gl_orig.run(img_gray_raw)
        raw = res[0] if res else ""
    except: raw = ""
    update_stats("FastPlate_Global (Base)", postprocess_plate(normalize_plate_text(raw)), gt, (time() - t0) * 1000)

    # 3. FastPlate_Finetuned
    if my_ort_session:
        t0 = time()
        try:
            # Делаем ресайз методом cv2.INTER_LINEAR (как было при обучении)
            img_input = cv2.resize(img_gray_raw, (140, 70), interpolation=cv2.INTER_LINEAR)

            # Динамически подстраиваемся под то, как экспортировалась модель (uint8 или float32)
            if 'float' in expected_type:
                img_input = img_input.astype(np.float32) / 255.0
            else:
                img_input = img_input.astype(np.uint8)

            img_input = np.expand_dims(img_input, axis=(0, -1))

            # Прогон
            input_name = my_ort_session.get_inputs()[0].name
            preds = my_ort_session.run(None, {input_name: img_input})[0]

            # Правильное CCE декодирование
            raw = cce_decode(preds, my_alphabet, my_pad_char)
        except Exception as e:
            raw = ""

        update_stats("FastPlate_Finetuned (New)", postprocess_plate(normalize_plate_text(raw)), gt, (time() - t0) * 1000)

# Вывод результатов
print("\n" + "="*50)
print("🏆 ИТОГОВЫЕ РЕЗУЛЬТАТЫ БЕНЧМАРКА 🏆")
print("="*50)

results_df = []
for key, v in bench.items():
    if v["total_plates"] == 0: continue
    pt = v["total_plates"]

    plate_acc = (v["plate_correct"] / pt) * 100
    char_acc = (v["char_correct"] / v["total_chars"]) * 100
    avg_time = (v["time_sum"] / pt)

    results_df.append({
        "Модель": key,
        "Plate Acc (Номера целиком)": f"{plate_acc:.2f}%",
        "Char Acc (Посимвольно)": f"{char_acc:.2f}%",
        "Время на кадр (ms)": f"{avg_time:.2f}"
    })

df = pd.DataFrame(results_df)
display(df)

Запуск бенчмарка на 500 изображениях...


Тестирование:   0%|          | 0/500 [00:00<?, ?it/s]


🏆 ИТОГОВЫЕ РЕЗУЛЬТАТЫ БЕНЧМАРКА 🏆


,Модель,Plate Acc (Номера целиком),Char Acc (Посимвольно),Время на кадр (ms)
0,RapidOCR,95.20%,98.18%,65.87
1,FastPlate_Global (Base),46.20%,81.79%,11.51
2,FastPlate_Finetuned (New),97.80%,98.82%,9.67
